# Build composable, interoperable multi-agent systems with A2A on AWS: enterprise identity (notebook 1 of 3)

> **Disclaimer:** This is sample code for demonstration and educational purposes only.
> It is not intended for production use without additional security review, testing, and hardening.

- Account: the enterprise account. Run it first.
- Time: about 3 min.
- You need: Bedrock access to Claude Sonnet 4.6 and Nova Lite in this account, in us-east-1, us-east-2, or us-west-2.
- You hand over: `onboarding-request.json` (public values only) to the SaaS provider for notebook 2.
- Services this notebook uses: AWS CloudFormation, Amazon Cognito, Amazon API Gateway, AWS Secrets Manager, Amazon Bedrock.
- Run the cells one at a time, top to bottom. Each step ends with a `PASS` line.

Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
SPDX-License-Identifier: MIT-0

## Run order

The demo runs in two AWS accounts in the same US Region. Run the three notebooks in order, each one
cell at a time, top to bottom.

| # | Account | Notebook | Time |
|---|---|---|---|
| 1 | Enterprise | `1_enterprise_identity.ipynb`: the enterprise identity; writes `onboarding-request.json` | about 3 min |
| 2 | SaaS provider | `2_saas_provider.ipynb`: this customer's AgentCore runtime; writes `onboarding-response.json` | about 5 min |
| 3 | Enterprise | `3_enterprise_agents.ipynb`: the agents, the end-to-end test, and the trust boundary checks | about 20 min |
| 4 | Both, after all testing | Clean up in notebook 3, then in notebook 2. Enterprise side first, so nothing calls the SaaS runtime while it is being removed | about 17 min |

Two files of public values move between the accounts. Neither file contains a secret.

<details><summary>Details</summary>

`onboarding-request.json` goes from notebook 1 to notebook 2, and `onboarding-response.json` from
notebook 2 to notebook 3. Copy the file into the other project folder, or paste the highlighted
line the notebook showed when it asks. They hold issuer URLs, client IDs, scopes, and endpoint URLs. In
production they move through the SaaS provider's onboarding portal or API.

</details>

## Architecture

Each organization's Cognito pool issues tokens only to its own workloads, and every client secret
stays in the account that created it; the other side trusts that pool by its public issuer, one
client ID, and one scope.

<details><summary>Diagram and trust table</summary>

```
Enterprise account                                       SaaS provider account
 Identity provider: Cognito a2a-enterprise-pool           Identity provider: Cognito a2a-saas-pool
   a2a-enterprise-orchestrator-saas  (secret stays here)    a2a-saas-callback-client (secret stays here)
   a2a-enterprise-orchestrator-user  (secret stays here)

 caller --JWT (enterprise pool, orchestrator-user)--> API Gateway user API
   --VPC link--> internal NLB :8000 --> Orchestrator (ECS, LangChain)
        | token from the ENTERPRISE pool (orchestrator-saas, scope a2a-enterprise-compliance-saas/invoke)
        +--HTTPS + JWT--> AgentCore runtime: trusts the enterprise pool only
                          (discoveryUrl, allowedClients, allowedScopes)
                          Compliance agent (Strands)
                            | token from the SAAS pool (callback client, scope a2a-saas-customer-callback/invoke)
                            | sent only to the callback URL registered at onboarding
 Context agent (ECS) <-- NLB :9001 <--VPC link-- API Gateway partner API <--HTTPS + JWT--+
   partner API trusts the SaaS pool only (issuer, callback client ID, route scope)

 What crosses accounts: two files of public values (onboarding-request.json, onboarding-response.json)
 and HTTPS calls with short-lived tokens. No secret, key, or IAM permission crosses.
```

There are no cross-account IAM roles, keys, or shared secrets; the trust works the way an IAM role
can trust an external OIDC identity provider.

| Call | Caller | Token issued by | Checked by |
|---|---|---|---|
| Agent Card fetch and A2A `message/send` to the compliance agent | Orchestrator (enterprise) | Enterprise pool, client `a2a-enterprise-orchestrator-saas`, scope `a2a-enterprise-compliance-saas/invoke` | AgentCore JWT authorizer: `discoveryUrl` (the enterprise pool), `allowedClients`, `allowedScopes` |
| A2A callback `message/send` to the context agent | Compliance agent (SaaS) | SaaS pool, client `a2a-saas-callback-client`, scope `a2a-saas-customer-callback/invoke` | Partner API JWT authorizer `saas-jwt`: SaaS issuer, callback client ID, route scope |
| Context agent's Agent Card | Anyone (the compliance agent checks it before it requests a callback token) | none | Public by design: a static discovery document with no model call |
| `POST /jobs`, `GET /jobs/{job_id}`, `GET /health`, `GET /agents` | Enterprise app (in the demo, notebook 3) | Enterprise pool, client `a2a-enterprise-orchestrator-user`, scope `a2a-enterprise-orchestrator/ask` | User API JWT authorizer `user-jwt`: issuer, client ID, route scope |

</details>

## Before you start

- Region: us-east-1, us-east-2, or us-west-2. The SaaS provider uses a different account in that Region.
- Bedrock model access here: Claude Sonnet 4.6 (`us.anthropic.claude-sonnet-4-6`) and Nova Lite
  (`us.amazon.nova-lite-v1:0`), which notebook 3's agents use. I0 checks it now.
- Nothing you copy from this notebook is secret.
- Where to run and the exact deployer permissions are in Reference at the end. A sandbox administrator
  role covers every notebook.

<details><summary>Which Regions this runs in</summary>

This demo runs in **us-east-1**, **us-east-2**, and **us-west-2**. Both accounts must use
the same one of those three. The models are reached through the `us.` cross-Region
inference profiles, which cover all three Regions, and Amazon Bedrock AgentCore is
available in all three. The configuration cell below uses the Region of your AWS profile
or notebook instance, so you only need to pick it there (or in `.env.enterprise`). This demo
is tested in us-east-1, us-east-2, and us-west-2, where both models' `us.` inference
profiles and AgentCore Runtime are available.

This notebook calls:

- Anthropic Claude Sonnet 4.6 (`us.anthropic.claude-sonnet-4-6`)
- Amazon Nova Lite (`us.amazon.nova-lite-v1:0`)

Running outside those US Regions needs more work and has not been tested: change the
model IDs to profiles for your geography (for example `eu.` or `apac.`) in the agent
files and in the Bedrock ARNs of the CloudFormation templates, widen the Region lists in
the onboarding checks and `SUPPORTED_REGIONS`, and confirm model access and AgentCore availability there. The
Bedrock preflight below checks model access in about ten seconds, before anything is
deployed.

</details>

## Setup

Installs this notebook's packages, loads the settings, and defines the helpers the steps use. Run
these cells first, and again after a kernel restart.

<details><summary>Details</summary>

Installs `boto3`, `requests`, and `python-dotenv`. The agents' own dependencies (LangChain, a2a-sdk, FastAPI) are installed only inside the images CodeBuild builds in notebook 3.

</details>

In [ ]:
# Install the notebook's own dependencies (floors and ceilings guard against breaking
# major-version upgrades; the toolkit is pinned exactly).
%pip install -q --no-cache-dir 'boto3>=1.42.1,<2.0.0' 'requests>=2.31.0,<3.0.0' 'python-dotenv>=1.0,<2.0'

pip may print a dependency-resolver ERROR or warning about preinstalled packages such as `sagemaker`
or `aiobotocore`. It is harmless here: the configuration cell checks the botocore it loaded.

<details><summary>Details</summary>

A Run All on a fresh kernel works: nothing imports boto3 before this install, and the configuration
cell below checks that the botocore it loaded is the one installed. If you ran other cells first, it
asks you to restart the kernel.

</details>

In [ ]:
# Find the project folder, load AWS settings from .env.enterprise (or .env), and print the
# side, account, and Region this notebook works in.
import base64
import json
import os
import re
import time
import uuid
from contextlib import contextmanager
from importlib.metadata import version as package_version
from pathlib import Path
from urllib.parse import unquote, urlsplit, urlunsplit


import boto3
import botocore
import requests
from botocore.config import Config
from botocore.exceptions import ClientError
from dotenv import dotenv_values, load_dotenv

if botocore.__version__ != package_version("botocore"):
    raise RuntimeError("packages were upgraded after boto3 was imported: restart the kernel, then run the cells "
                       "again from the top")


def find_project_folder(marker):
    """The project folder, the one that holds `marker`. In order: the notebook's own folder (if the
    kernel knows the notebook's path), the current folder, then the one subfolder that holds it.
    Stops if more than one subfolder does, rather than guess."""
    here = Path.cwd()
    own = globals().get("__vsc_ipynb_file__") or os.environ.get("JPY_SESSION_NAME")
    if own:
        nb_dir = (here / own).parent
        if (nb_dir / marker).is_file():
            return nb_dir
    if (here / marker).is_file():
        return here
    found = sorted(d for d in here.iterdir() if d.is_dir() and (d / marker).is_file())
    if len(found) == 1:
        print(f"Using the project folder {found[0].name} under {here}")
        return found[0]
    if found:
        raise RuntimeError(f"Project files found in more than one folder under {here}: "
                           f"{', '.join(d.name for d in found)}. Open the notebook from the folder you want "
                           f"to use, or remove the other copies.")
    raise RuntimeError(f"Project files not found under {here}. Open this notebook from the unzipped project folder.")


os.chdir(find_project_folder("cloudformation/enterprise-identity.yaml"))
HANDOFF_DIR = Path.cwd()  # the onboarding files are read and written here
print(f"Working directory: {HANDOFF_DIR}")

DOTENV = ".env.enterprise"
if Path(DOTENV).is_file():
    load_dotenv(DOTENV, override=True)  # the side's own file is explicit intent
    print(f"Loaded settings from {DOTENV}")
elif Path(".env").is_file():
    load_dotenv(".env", override=False)
    print("Loaded settings from .env")

# Every AWS client in this notebook: explicit timeouts, standard retries with jittered backoff, 5 attempts in all.
BOTO_CONFIG = Config(connect_timeout=3.1, read_timeout=60, retries={"mode": "standard", "total_max_attempts": 5})

profile = os.environ.get("AWS_PROFILE")
if profile and os.environ.get("AWS_ACCESS_KEY_ID"):
    raise RuntimeError(f"static keys in the environment would override the profile in {DOTENV}: unset them")
try:
    REGION = (os.environ.get("AWS_REGION") or os.environ.get("AWS_DEFAULT_REGION")
              or boto3.Session(profile_name=profile).region_name or "us-east-1")
    boto3.setup_default_session(profile_name=profile, region_name=REGION)  # profile None = default chain
    os.environ["AWS_DEFAULT_REGION"] = REGION  # so the agentcore subprocess agrees
    _identity = boto3.client("sts", region_name=REGION, config=BOTO_CONFIG).get_caller_identity()
except (botocore.exceptions.BotoCoreError, ClientError) as e:
    if profile:
        raise RuntimeError(f"Could not get AWS credentials for profile {profile} (AWS_PROFILE, from {DOTENV} or "
                           f"the environment): {type(e).__name__}. For an IAM Identity Center profile, run "
                           f"`aws sso login --profile {profile}` in a terminal, then re-run this cell.") from e
    raise RuntimeError(f"Could not get AWS credentials for the instance or default credentials: "
                       f"{type(e).__name__}. Check the notebook instance's IAM role.") from e
ACCOUNT_ID = _identity["Account"]
PARTITION = _identity["Arn"].split(":")[1]
SUPPORTED_REGIONS = ("us-east-1", "us-east-2", "us-west-2")

print("Side:    enterprise (notebook 1 of 3: identity)")
print(f"Account: {ACCOUNT_ID}")
print(f"Caller:  {_identity['Arn']}")
print(f"Profile: {profile or 'instance or default credentials'}")
print(f"Region:  {REGION}")

# Names (prefix a2a-enterprise shows which side owns each resource)
NAME_PREFIX = "a2a-enterprise"
IDENTITY_STACK = "a2a-enterprise-identity"
ORCHESTRATOR_SECRET_NAME = "a2a-enterprise/orchestrator-credentials"  # nosec B105  # a secret's name, not its value
REQUEST_FILE = "onboarding-request.json"
RESPONSE_FILE = "onboarding-response.json"

### Helpers

Every step below runs inside `with step(...)`. It prints a start line, ends with a `PASS` line and
its time, or prints `FAILED`, the reason, and a hint, then stops.

<details><summary>Details</summary>

The step body runs at notebook scope, so the values it assigns stay available to later cells.

| Helper | What it does |
|---|---|
| `log(msg, level)` | Timestamped line: `[HH:MM:SS] LEVEL msg` |
| `step(label, name, hint)` | Fail-fast wrapper for the steps. The body's last line is the step's `PASS` line; then the wrapper prints `done in` and the time |
| `cleanup(label, name)` | Wrapper for clean-up steps: reports an error and moves on, so one stuck resource does not stop the rest of the clean-up |
| `deploy_stack(name, template, parameters)` | Refuses a stack that already exists, creates it, waits, prints the failed resources on error, and returns the outputs |
| `wait_text(max_minutes, usual)` | The wait line of a stack create or delete: the usual time, if known, then the timeout |
| `refuse_other_side_account(stacks, dotenv)` | Stops if this account holds the other side's stacks, or if the other side's `.env` file names the same profile or account |
| `http(method, url, ...)` | Every HTTP call in this notebook: no redirects, 30-second default timeout |
| `request_ids(r)` | A response's `www-authenticate`, `apigw-requestid`, and `x-amzn-requestid` headers, for the log lines that show who answered |
| `client_credentials_token(...)` | OAuth 2.0 client-credentials grant against a Cognito token endpoint. Returns the token; never prints it |
| `own_client_token(...)` | Same, for an app client in this account: reads its secret in memory with `describe_user_pool_client` and discards it |
| `pinned_agent_url(card_url, card_url_field)` | The same card-pinning rule the orchestrator applies |
| `load_handoff(file, kind, fields, step_label)` | Reads an onboarding file next to this notebook, or asks for the one-line paste if the file isn't there. Says so plainly if it gets the other notebook's handoff. The only prompt in any notebook |
| `check_discovery(issuer, token_endpoint)` | Fetches the issuer's OIDC discovery document (capped, no redirects) and checks `issuer`, `jwks_uri`, and `token_endpoint` |

</details>

In [ ]:
# Logging, fail-fast steps, and CloudFormation, S3, and log helpers (standard library plus boto3).
cf = boto3.client("cloudformation", region_name=REGION, config=BOTO_CONFIG)
s3 = boto3.client("s3", region_name=REGION, config=BOTO_CONFIG)
logs_client = boto3.client("logs", region_name=REGION, config=BOTO_CONFIG)
secrets = boto3.client("secretsmanager", region_name=REGION, config=BOTO_CONFIG)


def log(msg, level="INFO"):
    """Timestamped log line: [HH:MM:SS] LEVEL msg"""
    print(f"[{time.strftime('%H:%M:%S')}] {level:5s} {msg}", flush=True)


def _fmt_elapsed(seconds):
    return f"{seconds / 60:.0f}m" if seconds >= 90 else f"{seconds:.1f}s"


@contextmanager
def step(label, name, hint=None):
    """Fail-fast deploy step: prints start, done, or FAILED plus a hint, and re-raises errors."""
    log(f"Step {label}  {name} ...")
    t0 = time.time()
    try:
        yield
    except BaseException as e:
        log(f"  FAILED: {type(e).__name__}: {e}  (after {_fmt_elapsed(time.time() - t0)})", "ERROR")
        if hint:
            log(f"  hint: {hint}", "ERROR")
        raise
    else:
        log(f"  done in {_fmt_elapsed(time.time() - t0)}")


CLEANUP_ERRORS = []


@contextmanager
def cleanup(label, name):
    """Clean-up step: a missing resource is fine; any other error is reported and the next step runs."""
    log(f"Clean up {label}  {name} ...")
    try:
        yield
    except Exception as e:
        CLEANUP_ERRORS.append(f"{label}: {type(e).__name__}: {e}")
        log(f"  ERROR: {type(e).__name__}: {e}. Continuing; the final check lists anything left.", "ERROR")
    else:
        log("  done")


def error_code(exc):
    return exc.response.get("Error", {}).get("Code", "") if isinstance(exc, ClientError) else ""


def show_stack_failures(stack_name):
    """Print the resources that failed in a stack (the real cause behind a waiter error)."""
    try:
        events = cf.describe_stack_events(StackName=stack_name)["StackEvents"]
    except ClientError as e:
        log(f"Could not read stack events for {stack_name}: {error_code(e)}", "WARN")
        return
    seen = set()
    for e in events:
        if "FAILED" not in e.get("ResourceStatus", "") or e.get("ResourceType") == "AWS::CloudFormation::Stack":
            continue
        if e["LogicalResourceId"] in seen:
            continue
        seen.add(e["LogicalResourceId"])
        print(f"    {e['LogicalResourceId']} ({e.get('ResourceType')}) [{e.get('ResourceStatus')}]: "
              f"{e.get('ResourceStatusReason', '(no reason reported)')}", flush=True)
    if not seen:
        log(f"No failed resources reported for {stack_name}.", "WARN")


def stack_status(stack_name):
    """The stack's status, or None if it does not exist."""
    try:
        return cf.describe_stacks(StackName=stack_name)["Stacks"][0]["StackStatus"]
    except ClientError as e:
        if error_code(e) == "ValidationError" and "does not exist" in str(e):
            return None
        raise


def stack_exists(stack_name):
    """True if the stack exists in any state other than DELETE_COMPLETE."""
    return stack_status(stack_name) not in (None, "DELETE_COMPLETE")


def refuse_existing_stack(stack_name):
    """Stop if the stack already exists: this notebook always creates its stacks fresh."""
    if stack_exists(stack_name):
        raise RuntimeError(
            f"Stack '{stack_name}' already exists in {REGION}. This notebook creates it fresh and "
            f"never reuses one. See 'If a step failed' in Reference for the Clean up to run, then re-run.")


def refuse_other_side_account(other_stacks, other_dotenv):
    """Stop if this looks like the other side's account, before anything is created.

    1. Any of the other side's stacks is here. 2. The other side's .env file names this same
    profile. 3. The other side's profile resolves (read-only STS call) to this account. With no
    other .env file (a notebook instance in each account), the printed side and account are the check.
    """
    found = [name for name in other_stacks if stack_exists(name)]
    if found:
        raise RuntimeError(
            f"Account {ACCOUNT_ID} has the other side's stack(s) {', '.join(found)}, so this looks like the wrong "
            f"account for this notebook. Notebooks 1 and 3 run in the enterprise account and notebook 2 in the "
            f"SaaS provider account. Switch the profile in {DOTENV}, or use the other notebook instance, then "
            f"run this notebook again from the top.")
    if not Path(other_dotenv).is_file():
        return
    other = dotenv_values(other_dotenv).get("AWS_PROFILE")
    if not other:
        log(f"{other_dotenv} names no AWS_PROFILE; the check against the other side is skipped", "WARN")
        return
    if other == profile:
        raise RuntimeError(f"{DOTENV} and {other_dotenv} name the same profile {other}; each side needs its own "
                           f"account's profile. Fix AWS_PROFILE in {DOTENV}, then run this notebook again from "
                           f"the top.")
    try:
        other_account = boto3.Session(profile_name=other).client(
            "sts", region_name=REGION, config=BOTO_CONFIG).get_caller_identity()["Account"]
    except Exception as e:
        log(f"Could not resolve the account of profile {other} ({other_dotenv}): {type(e).__name__}; the check "
            f"against the other side is skipped", "WARN")
        return
    if other_account == ACCOUNT_ID:
        raise RuntimeError(f"The profiles in {DOTENV} ({profile or 'default'}) and {other_dotenv} ({other}) both "
                           f"resolve to account {ACCOUNT_ID}. The demo needs two different accounts: fix "
                           f"AWS_PROFILE in one of the files, then run this notebook again from the top.")
    log(f"Other side: profile {other} ({other_dotenv}) is account {other_account}, not this one")


def wait_for_stack(stack_name, operation, max_minutes):
    cf.get_waiter(f"stack_{operation}_complete").wait(
        StackName=stack_name, WaiterConfig={"Delay": 30, "MaxAttempts": max_minutes * 2})


def get_stack_outputs(stack_name):
    stack = cf.describe_stacks(StackName=stack_name)["Stacks"][0]
    return {o["OutputKey"]: o["OutputValue"] for o in stack.get("Outputs", [])}


def wait_text(max_minutes, usual=None):
    """The wait shown while a stack is created or deleted: the usual time (if measured), then the timeout."""
    return f"usually {usual}; waits up to {max_minutes} min" if usual else f"waits up to {max_minutes} min"


def deploy_stack(stack_name, template_file, parameters, max_minutes=30, usual=None):
    """Create a stack fresh (refusing an existing one), wait for it, and return its outputs."""
    refuse_existing_stack(stack_name)
    try:
        cf.create_stack(
            StackName=stack_name,
            TemplateBody=Path(template_file).read_text(),
            Parameters=[{"ParameterKey": k, "ParameterValue": v} for k, v in parameters.items()],
            Capabilities=["CAPABILITY_IAM"],
            ClientRequestToken=f"a2a-{uuid.uuid4().hex}",  # a retried create is the same request, not a second one
        )
        log(f"  creating {stack_name} ({wait_text(max_minutes, usual)}) ...")
        wait_for_stack(stack_name, "create", max_minutes)
    except Exception:
        show_stack_failures(stack_name)
        raise
    return get_stack_outputs(stack_name)


def delete_stack_and_wait(stack_name, max_minutes=30, usual=None):
    if not stack_exists(stack_name):
        log(f"  {stack_name}: not present")
        return
    cf.delete_stack(StackName=stack_name)
    log(f"  deleting {stack_name} ({wait_text(max_minutes, usual)}) ...")
    try:
        wait_for_stack(stack_name, "delete", max_minutes)
    except Exception:
        show_stack_failures(stack_name)
        raise
    log(f"  {stack_name}: deleted")


def secret_exists(name):
    """True if the secret exists, including one scheduled for deletion."""
    try:
        secrets.describe_secret(SecretId=name)
        return True
    except ClientError as e:
        if error_code(e) == "ResourceNotFoundException":
            return False
        raise


def bucket_exists(bucket):
    """True if this account owns the bucket. Raises if the name is taken or not readable."""
    try:
        s3.head_bucket(Bucket=bucket, ExpectedBucketOwner=ACCOUNT_ID)
        return True
    except ClientError as e:
        if error_code(e) in ("404", "NoSuchBucket", "NotFound"):
            return False
        raise RuntimeError(f"Bucket {bucket} exists but is not readable as this account's bucket "
                           f"({error_code(e)}); it may belong to another account.") from None


def delete_objects_with_prefix(bucket, prefix):
    """Delete every object under a prefix (unversioned bucket). Returns the count."""
    deleted, token = 0, None
    while True:
        kwargs = {"Bucket": bucket, "Prefix": prefix, "ExpectedBucketOwner": ACCOUNT_ID}
        if token:
            kwargs["ContinuationToken"] = token
        page = s3.list_objects_v2(**kwargs)
        keys = [{"Key": o["Key"]} for o in page.get("Contents", [])]
        if keys:
            s3.delete_objects(Bucket=bucket, Delete={"Objects": keys, "Quiet": True}, ExpectedBucketOwner=ACCOUNT_ID)
            deleted += len(keys)
        token = page.get("NextContinuationToken")
        if not token:
            return deleted


def log_group_names(prefix):
    names, token = [], None
    while True:
        kwargs = {"logGroupNamePrefix": prefix}
        if token:
            kwargs["nextToken"] = token
        page = logs_client.describe_log_groups(**kwargs)
        names += [g["logGroupName"] for g in page.get("logGroups", [])]
        token = page.get("nextToken")
        if not token:
            return names


def delete_log_group(name):
    try:
        logs_client.delete_log_group(logGroupName=name)
        log(f"  {name}: deleted")
    except logs_client.exceptions.ResourceNotFoundException:
        log(f"  {name}: not present")


if REGION not in SUPPORTED_REGIONS:
    log(f"Region {REGION} is outside the tested set {SUPPORTED_REGIONS}; see 'Which Regions this runs in'.", "WARN")
print("Helpers loaded: log, step, cleanup, deploy_stack, delete_stack_and_wait")

In [ ]:
# OAuth and HTTP helpers. Secrets and tokens are passed as arguments and never printed.
cognito = boto3.client("cognito-idp", region_name=REGION, config=BOTO_CONFIG)


def http(method, url, **kw):
    """Every HTTP call in this notebook goes through here: no redirects, so a 3xx can't move a
    credential to another host, and a 30-second default timeout."""
    kw.setdefault("timeout", 30)
    return requests.request(method, url, allow_redirects=False, **kw)


def client_credentials_token(token_endpoint, client_id, client_secret, scope, attempts=1, wait_seconds=15):
    """Return (access_token, expires_in) from a Cognito token endpoint. Raises on failure."""
    last = "no response"
    for attempt in range(1, attempts + 1):
        try:
            r = http("POST", token_endpoint, data={"grant_type": "client_credentials", "scope": scope},
                     auth=(client_id, client_secret), timeout=15)
        except requests.RequestException as e:
            last = type(e).__name__
        else:
            try:
                body = r.json()
            except ValueError:
                body = {}
            body = body if isinstance(body, dict) else {}
            if r.status_code == 200 and body.get("access_token"):
                return body["access_token"], body.get("expires_in")
            # Cognito's error body has only a code such as invalid_client or invalid_scope.
            last = f"HTTP {r.status_code} {body.get('error', '')}".strip()
        if attempt < attempts:
            time.sleep(wait_seconds)
    raise RuntimeError(f"Token request to {token_endpoint} failed: {last}")


def own_client_token(user_pool_id, client_id, token_endpoint, scope, attempts=1):
    """Token for an app client in THIS account. Its secret is read in memory only and discarded."""
    client_secret = cognito.describe_user_pool_client(
        UserPoolId=user_pool_id, ClientId=client_id)["UserPoolClient"]["ClientSecret"]
    try:
        return client_credentials_token(token_endpoint, client_id, client_secret, scope, attempts)
    finally:
        del client_secret


def token_claims(token):
    """Decode a JWT's claims for display (no signature check; the authorizers verify it)."""
    payload = token.split(".")[1]
    return json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))


def check_token_claims(token, client_id, scope):
    """The claims an authorizer will check: client_id, scope, and token_use. Returns the claims."""
    claims = token_claims(token)
    if (claims.get("client_id") != client_id or scope not in str(claims.get("scope", "")).split()
            or claims.get("token_use") != "access"):
        raise RuntimeError("The token does not carry the expected client_id, scope, and token_use")
    return claims


def wait_for_cognito_domain(domain, minutes=5):
    """Wait for a new Cognito prefix domain to report ACTIVE (its token endpoint lags creation)."""
    for _ in range(minutes * 4):
        status = cognito.describe_user_pool_domain(Domain=domain).get("DomainDescription", {}).get("Status")
        if status == "ACTIVE":
            return
        time.sleep(15)
    log(f"Cognito domain {domain} is not ACTIVE yet; the token request below retries.", "WARN")


_WELL_KNOWN_CARD_SUFFIX = "/.well-known/agent-card.json"


def pinned_agent_url(card_url, card_agent_url):
    """The orchestrator's card-pinning rule (agents/enterprise_orchestrator.py).

    The card's `url` must equal the card URL minus /.well-known/agent-card.json: same
    scheme and host, paths equal after percent-decoding and ignoring trailing slashes,
    same query, no fragments. Returns the base to send A2A messages to, or None.
    """
    if not isinstance(card_url, str) or not isinstance(card_agent_url, str) or not card_agent_url:
        return None
    card = urlsplit(card_url)
    agent = urlsplit(card_agent_url)
    if card.scheme.lower() != "https" or not card.netloc or not card.path.endswith(_WELL_KNOWN_CARD_SUFFIX):
        return None
    base_path = card.path[: -len(_WELL_KNOWN_CARD_SUFFIX)]
    if agent.scheme.lower() != card.scheme.lower() or agent.netloc.lower() != card.netloc.lower():
        return None
    if unquote(agent.path).rstrip("/") != unquote(base_path).rstrip("/"):
        return None
    if agent.query != card.query or card.fragment or agent.fragment:
        return None
    return urlunsplit((card.scheme, card.netloc, base_path.rstrip("/") + "/", card.query, ""))


def check_card_security(card, token_url, scope):
    """The card must declare an OAuth 2.0 client-credentials scheme with this token URL and scope."""
    for scheme in (card.get("securitySchemes") or {}).values():
        scheme = scheme if isinstance(scheme, dict) else {}
        flow = (scheme.get("flows") or {}).get("clientCredentials") or {}
        if scheme.get("type") == "oauth2" and flow.get("tokenUrl") == token_url and scope in (flow.get("scopes") or {}):
            return
    raise RuntimeError(f"The Agent Card does not declare an OAuth2 client-credentials scheme for {token_url} with scope {scope}")


def a2a_message(text):
    """An A2A v0.3 JSON-RPC message/send request body."""
    return {"jsonrpc": "2.0", "id": f"nb-{uuid.uuid4().hex}", "method": "message/send",
            "params": {"message": {"role": "user", "parts": [{"kind": "text", "text": text}],
                                   "messageId": uuid.uuid4().hex}}}


def a2a_text(result):
    """Text parts from an A2A v0.3 result (artifacts first, then agent history)."""
    texts = [p.get("text", "") for a in result.get("artifacts") or [] for p in a.get("parts") or []
             if p.get("kind") == "text"]
    if not texts:
        texts = [p.get("text", "") for m in result.get("history") or [] if m.get("role") == "agent"
                 for p in m.get("parts") or [] if p.get("kind") == "text"]
    return "".join(texts)


def request_ids(r):
    """The response headers that show who answered: www-authenticate and both AWS request IDs
    (never the request's credentials)."""
    return (f"www-authenticate={r.headers.get('www-authenticate', '-')!r}, "
            f"apigw-requestid={r.headers.get('apigw-requestid', '-')}, "
            f"x-amzn-requestid={r.headers.get('x-amzn-requestid', '-')}")


def denial_evidence(r):
    """Status plus the headers that show who answered (never the request's credentials)."""
    return f"HTTP {r.status_code}, {request_ids(r)}"


def check_one_reader_policy(secret_id, role_arn):
    """Read the secret's resource policy (never its value). It must be one Deny on GetSecretValue
    for every principal whose ARN is not role_arn."""
    raw = secrets.get_resource_policy(SecretId=secret_id).get("ResourcePolicy") or "{}"
    statements = json.loads(raw).get("Statement") or []
    statements = statements if isinstance(statements, list) else [statements]
    s = statements[0] if len(statements) == 1 and isinstance(statements[0], dict) else {}
    actions = s.get("Action") if isinstance(s.get("Action"), list) else [s.get("Action")]
    principal_arn = ((s.get("Condition") or {}).get("ArnNotEquals") or {}).get("aws:PrincipalArn")
    if (s.get("Effect") != "Deny" or actions != ["secretsmanager:GetSecretValue"]
            or s.get("Principal") not in ("*", {"AWS": "*"}) or principal_arn not in (role_arn, [role_arn])):
        raise RuntimeError(f"The resource policy on {secret_id} is not the single Deny on GetSecretValue "
                           f"for every principal except {role_arn}")


print("OAuth and HTTP helpers loaded")

### Onboarding handoff helpers

The onboarding files hold public values only, and every field is checked before anything is deployed.

<details><summary>Details</summary>

Each file is read from next to this notebook, or pasted as one line when the file isn't there, and
then checked field by field:

- every URL is https, with no user info, port, query, or fragment
- the issuer, token endpoint, callback URL, and Agent Card URL each match their exact
  AWS shape, in us-east-1, us-east-2, or us-west-2, all in one Region and in this
  notebook's Region
- the issuer's OIDC discovery document is fetched (capped at 64 KiB, no redirects) and
  must name the same issuer, an https `jwks_uri` on the issuer's host, and the same token
  endpoint
- values that point at this account are refused where that can be determined

</details>

In [ ]:
# Onboarding request and response: load, validate, and write (public values only).
REQUEST_KIND, RESPONSE_KIND = "a2a-onboarding-request", "a2a-onboarding-response"
REQUEST_FIELDS = ("issuer", "discovery_url", "token_endpoint", "client_id", "scope", "callback_url")
RESPONSE_FIELDS = ("saas_agent_card_url", "issuer", "token_endpoint", "callback_client_id", "callback_scope",
                   "registered_client_id", "registered_callback_url")
HANDOFF_MAX_CHARS = 4096
DISCOVERY_MAX_BYTES = 65536
# What each handoff is and which notebook writes it, for the message when the other one is given.
HANDOFF_WRITERS = {REQUEST_KIND: ("onboarding request", "notebook 1"),
                   RESPONSE_KIND: ("onboarding response", "notebook 2")}

_R = r"(us-east-1|us-east-2|us-west-2)"
ISSUER_RE = re.compile(r"https://cognito-idp\." + _R + r"\.amazonaws\.com/" + _R + r"_[0-9A-Za-z]+")
TOKEN_ENDPOINT_RE = re.compile(r"https://([a-z0-9-]{1,63})\.auth\." + _R + r"\.amazoncognito\.com/oauth2/token")
CALLBACK_URL_RE = re.compile(r"https://([a-z0-9]{10})\.execute-api\." + _R + r"\.amazonaws\.com/")
CARD_URL_RE = re.compile(r"https://bedrock-agentcore\." + _R
                         + r"\.amazonaws\.com/runtimes/([^/]+)/invocations/\.well-known/agent-card\.json")
RUNTIME_ARN_RE = re.compile(r"arn:aws[a-z-]*:bedrock-agentcore:" + _R + r":(\d{12}):runtime/[A-Za-z0-9_-]+")
CLIENT_ID_RE = re.compile(r"[0-9a-zA-Z]{1,128}")
SCOPE_RE = re.compile(r"[a-z0-9-]{1,64}/[a-z0-9-]{1,64}")


class DiscoveryUnavailable(ValueError):
    """The issuer's OIDC discovery document could not be fetched (non-200 or no connection)."""


def load_handoff(filename, kind, fields, step_label=None):
    """Read an onboarding file next to this notebook, or ask for the one-line paste if it isn't there.

    Returns (values, pasted). Checks the size, whether it is the other notebook's handoff, the
    exact keys, kind, and version; the caller validates the values, then writes a pasted handoff
    to the file with write_handoff(). S1 and E1 pass their step_label; both run before anything
    is deployed.
    """
    path = HANDOFF_DIR / filename
    pasted = not path.is_file()
    if pasted:
        text = input(f"{filename} is not in {HANDOFF_DIR}. Paste the one-line {kind} JSON: ")
    else:
        with path.open(encoding="utf-8-sig") as f:  # also accepts a file an editor saved with a BOM
            text = f.read(HANDOFF_MAX_CHARS + 1)
        log(f"Read {path}")
    if len(text) > HANDOFF_MAX_CHARS:
        raise ValueError(f"The {kind} is longer than {HANDOFF_MAX_CHARS} characters: use the file or the "
                         f"one line the other notebook printed")
    try:
        data = json.loads(text)
    except ValueError:
        raise ValueError(f"The {kind} must be the one-line JSON object the other notebook printed") from None
    got = data.get("kind") if isinstance(data, dict) else None
    if isinstance(got, str) and got != kind and got in HANDOFF_WRITERS:
        # The other notebook's handoff, for example the request pasted at E1: say so before the keys check.
        (got_name, got_writer), (want_name, want_writer) = HANDOFF_WRITERS[got], HANDOFF_WRITERS[kind]
        reader = f"{step_label} needs" if step_label else "This cell needs"
        raise ValueError(f"This is the {got_name} from {got_writer} (kind {got}). {reader} the {want_name} that "
                         f"{want_writer} wrote (kind {kind})." + (" Nothing is deployed yet." if step_label else ""))
    expected = {"kind", "version", *fields}
    if not isinstance(data, dict) or set(data) != expected:
        raise ValueError(f"The {kind} must have exactly these keys: {', '.join(sorted(expected))}")
    if data["kind"] != kind:
        raise ValueError(f"Expected kind {kind!r}, got {data['kind']!r}: check which file or line you used")
    if not (type(data["version"]) is int and data["version"] == 1):
        raise ValueError(f"Unsupported {kind} version {data['version']!r}: this notebook reads version 1")
    if not all(isinstance(data[k], str) and data[k] for k in fields):
        raise ValueError(f"Every value in the {kind} must be a non-empty string")
    return data, pasted


def write_handoff(filename, data):
    """Write a handoff file next to this notebook (public values only)."""
    path = HANDOFF_DIR / filename
    path.write_text(json.dumps(data, indent=2) + "\n", encoding="utf-8")
    log(f"Wrote {path}")


def https_url_parts(url, field):
    """Split an https URL that has no user info, port, query, or fragment."""
    if not isinstance(url, str) or any(c.isspace() for c in url) or '"' in url or "\\" in url:
        raise ValueError(f"{field} must be a URL with no spaces, quotes, or backslashes")
    parts = urlsplit(url)
    if parts.scheme != "https" or not parts.hostname or parts.username or parts.password or parts.port:
        raise ValueError(f"{field} must be an https URL with no user info or port")
    if parts.query or parts.fragment:
        raise ValueError(f"{field} must not have a query or fragment")
    return parts


def _match(pattern, value, field, shape):
    m = pattern.fullmatch(value)
    if not m:
        raise ValueError(f"{field} must be {shape} in us-east-1, us-east-2, or us-west-2")
    return m


def _issuer(value, field="issuer"):
    https_url_parts(value, field)
    m = _match(ISSUER_RE, value, field, "https://cognito-idp.<region>.amazonaws.com/<region>_<pool ID>")
    if m.group(1) != m.group(2):
        raise ValueError(f"{field}: the pool ID's Region must equal the host's Region")
    return m.group(1), value.rsplit("/", 1)[1]


def _token_endpoint(value, field="token_endpoint"):
    https_url_parts(value, field)
    m = _match(TOKEN_ENDPOINT_RE, value, field, "https://<prefix>.auth.<region>.amazoncognito.com/oauth2/token")
    return m.group(2), m.group(1)


def _callback_url(value, field="callback_url"):
    https_url_parts(value, field)
    m = _match(CALLBACK_URL_RE, value, field, "https://<10-character API ID>.execute-api.<region>.amazonaws.com/")
    return m.group(2), m.group(1)


def _ids_and_scopes(data, id_fields, scope_fields):
    for f in id_fields:
        if not CLIENT_ID_RE.fullmatch(data[f]):
            raise ValueError(f"{f} must be 1 to 128 letters and digits (a Cognito app client ID)")
    for f in scope_fields:
        if not SCOPE_RE.fullmatch(data[f]):
            raise ValueError(f"{f} must look like resource-server/scope (lowercase letters, digits, hyphens)")


def _one_region(regions, kind):
    if len(set(regions)) != 1:
        raise ValueError(f"Every URL in the {kind} must be in the same Region; found {sorted(set(regions))}")
    region = regions[0]
    if region != REGION:
        raise ValueError(f"The {kind} is for {region}, and this notebook runs in {REGION}. Both accounts must use "
                         f"the same Region: set AWS_REGION in {DOTENV} (or use a notebook instance in {region}), "
                         f"then run this notebook again from the top. This step has deployed nothing.")
    return region


def validate_request(data, own_account):
    """Field checks on an onboarding request. own_account None skips the own-account check (self-check)."""
    issuer_region, pool_id = _issuer(data["issuer"])
    https_url_parts(data["discovery_url"], "discovery_url")
    if data["discovery_url"] != data["issuer"] + "/.well-known/openid-configuration":
        raise ValueError("discovery_url must be exactly the issuer followed by /.well-known/openid-configuration")
    token_region, token_prefix = _token_endpoint(data["token_endpoint"])
    callback_region, api_id = _callback_url(data["callback_url"])
    _ids_and_scopes(data, ["client_id"], ["scope"])
    region = _one_region([issuer_region, token_region, callback_region], "onboarding request")
    if own_account and token_prefix == f"a2a-enterprise-{own_account}":
        raise ValueError("The request's token endpoint belongs to this AWS account. Notebook 2 must run in the "
                         "SaaS provider account, and notebook 1 in the enterprise account.")
    return {"region": region, "pool_id": pool_id, "api_id": api_id}


def validate_response(data, own_account, own_ids):
    """Field checks on an onboarding response, including the registered_* echoes.

    own_account None skips the own-account checks (self-check). own_ids holds this
    enterprise's caller client ID and callback URL, which the SaaS provider must have registered.
    """
    https_url_parts(data["saas_agent_card_url"], "saas_agent_card_url")
    card = _match(CARD_URL_RE, data["saas_agent_card_url"], "saas_agent_card_url",
                  "https://bedrock-agentcore.<region>.amazonaws.com/runtimes/<URL-encoded runtime ARN>"
                  "/invocations/.well-known/agent-card.json")
    runtime_arn = unquote(card.group(2))
    arn = RUNTIME_ARN_RE.fullmatch(runtime_arn)
    if not arn:
        raise ValueError("saas_agent_card_url must contain the URL-encoded AgentCore runtime ARN")
    issuer_region, pool_id = _issuer(data["issuer"])
    token_region, token_prefix = _token_endpoint(data["token_endpoint"])
    registered_region, _ = _callback_url(data["registered_callback_url"], "registered_callback_url")
    _ids_and_scopes(data, ["callback_client_id", "registered_client_id"], ["callback_scope"])
    region = _one_region([card.group(1), arn.group(1), issuer_region, token_region, registered_region],
                         "onboarding response")
    if own_account and arn.group(2) == own_account:
        raise ValueError(f"The SaaS runtime in this response is in this AWS account ({own_account}). Notebook 3 "
                         "must run in the enterprise account, and notebook 2 in the SaaS provider account.")
    if own_account and token_prefix == f"a2a-saas-{own_account}":
        raise ValueError("The response's token endpoint belongs to this AWS account. Notebook 3 must run in the "
                         "enterprise account, and notebook 2 in the SaaS provider account.")
    if (data["registered_client_id"] != own_ids["client_id"]
            or data["registered_callback_url"] != own_ids["callback_url"]):
        raise ValueError("The response registers a different caller client or callback URL than this enterprise's "
                         "identity (notebook 1): it is stale or meant for another customer. Ask the SaaS provider "
                         "to run notebook 2 with this enterprise's onboarding-request.json, then replace "
                         "onboarding-response.json.")
    return {"region": region, "runtime_arn": runtime_arn, "saas_account": arn.group(2), "pool_id": pool_id}


def check_discovery(issuer, token_endpoint):
    """Fetch the issuer's OIDC discovery document and check it against the handoff values.

    The URL is always derived from the issuer, never taken from the handoff. The read is
    capped at 64 KiB, and the http() helper refuses redirects.
    """
    discovery_url = issuer + "/.well-known/openid-configuration"
    try:
        r = http("GET", discovery_url, stream=True, timeout=10, headers={"Accept": "application/json"})
    except requests.RequestException as e:
        raise DiscoveryUnavailable(f"The issuer's discovery document is unavailable ({type(e).__name__}): "
                                   f"{discovery_url}") from None
    try:
        if r.status_code != 200:
            raise DiscoveryUnavailable(f"The issuer's discovery document is unavailable (HTTP {r.status_code}): "
                                       f"{discovery_url}")
        body = r.raw.read(DISCOVERY_MAX_BYTES + 1, decode_content=True)
    finally:
        r.close()
    if len(body) > DISCOVERY_MAX_BYTES:
        raise ValueError(f"The discovery document at {discovery_url} is larger than {DISCOVERY_MAX_BYTES} bytes")
    try:
        doc = json.loads(body)
    except ValueError:
        doc = None
    if not isinstance(doc, dict):
        raise ValueError(f"The discovery document at {discovery_url} is not a JSON object")
    if doc.get("issuer") != issuer:
        raise ValueError(f"The discovery document names issuer {doc.get('issuer')!r}, not {issuer}")
    jwks = https_url_parts(doc.get("jwks_uri"), "jwks_uri")
    if jwks.hostname != urlsplit(issuer).hostname:
        raise ValueError("The discovery document's jwks_uri is not on the issuer's host")
    if "token_endpoint" in doc:
        if doc["token_endpoint"] != token_endpoint:
            raise ValueError(f"The discovery document names token endpoint {doc['token_endpoint']!r}, "
                             f"not {token_endpoint}")
    else:
        log("The discovery document lists no token_endpoint; the token endpoint pattern check alone applies.")
    log(f"OIDC discovery checked: issuer {issuer}, jwks_uri on {jwks.hostname}")


def own_account_checks(checks, hint):
    """Refuse a handoff value that names a resource in this account, where that can be determined.

    checks: (what, service, region, method, kwargs). A call that succeeds means the resource
    is in this account. Not-found passes; any other error is logged as inconclusive.
    """
    for what, service, region, method, kwargs in checks:
        try:
            getattr(boto3.client(service, region_name=region, config=BOTO_CONFIG), method)(**kwargs)
        except ClientError as e:
            if error_code(e) in ("ResourceNotFoundException", "NotFoundException"):
                log(f"  own-account check: {what} is not in this account")
            else:
                log(f"  own-account check inconclusive ({error_code(e) or type(e).__name__}) for {what}; continuing",
                    "WARN")
            continue
        except Exception as e:
            log(f"  own-account check inconclusive ({type(e).__name__}) for {what}; continuing", "WARN")
            continue
        raise ValueError(f"{what} is in this AWS account ({ACCOUNT_ID}). {hint}")


print("Onboarding helpers loaded")

## I0. Preflight: Bedrock model access

Sends a one-token request to each model this account needs, before anything is deployed.

Pass when: every model answers, and the step logs `PASS I0`.

<details><summary>Details</summary>

Model access is per account, so each side checks its own models. Notebook 3's agents use these models; checking now finds a model-access gap before the SaaS provider deploys anything. The step sends the
smallest possible `Converse` request (one output token) to each model:

- Anthropic Claude Sonnet 4.6 (`us.anthropic.claude-sonnet-4-6`)
- Amazon Nova Lite (`us.amazon.nova-lite-v1:0`)

Each is a cross-Region inference profile (the `us.` prefix is required). In commercial
Regions, Amazon Bedrock enables a model on its first call, so there is no access request
to submit. If a model fails, check what that first call needs: for Anthropic models, the
first-time use form for this account; a valid payment method on the account; and
`aws-marketplace:Subscribe`, `aws-marketplace:Unsubscribe`, and
`aws-marketplace:ViewSubscriptions` for this notebook's role. See
[Amazon Bedrock model access](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html).

</details>

In [ ]:
with step("I0", "Preflight: Bedrock model access",
          hint="If the error names the Region, see 'Which Regions this runs in'; otherwise check the first-call "
               "needs: the Anthropic first-time use form, a payment method, and the aws-marketplace permissions "
               "in Reference. The us. prefix needs cross-Region inference. Then re-run this cell."):
    if REGION not in SUPPORTED_REGIONS:
        raise RuntimeError(f"This demo runs in {', '.join(SUPPORTED_REGIONS)}; this notebook is in {REGION}. Set "
                           f"AWS_REGION in {DOTENV}, or use a notebook instance in one of those Regions, then run "
                           f"this notebook again from the top.")
    bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION, config=BOTO_CONFIG)
    PREFLIGHT_MODELS = {'us.anthropic.claude-sonnet-4-6': 'Enterprise Orchestrator', 'us.amazon.nova-lite-v1:0': 'Enterprise Context Agent'}
    failed = []
    for model_id, used_by in PREFLIGHT_MODELS.items():
        try:
            bedrock_runtime.converse(
                modelId=model_id,
                messages=[{"role": "user", "content": [{"text": "ping"}]}],
                inferenceConfig={"maxTokens": 1},
            )
            log(f"  {model_id} ({used_by}): invokable")
        except Exception as e:
            failed.append(model_id)
            log(f"  {model_id} ({used_by}): FAILED {type(e).__name__} {error_code(e)}", "ERROR")
    if failed:
        print(f"Model access prerequisites for {', '.join(failed)}: "
              "https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html")
        raise RuntimeError(f"Bedrock model access preflight failed for {', '.join(failed)}")
    log(f"PASS I0: Bedrock model access works in {REGION} for {', '.join(PREFLIGHT_MODELS)}")

### I0.1. Preflight: the enterprise account, then leftovers from an earlier run

Checks that this is the enterprise account, then that nothing from an earlier run is left.

Pass when: the step logs `PASS I0.1`.

<details><summary>Details</summary>

The account check stops if the SaaS provider's stacks (`a2a-saas-identity`, `a2a-saas-runtime`) are
here, or, on a local Jupyter with both `.env.enterprise` and `.env.saas`, if both files name the same
profile or the SaaS profile resolves to this account (a read-only STS call).

This notebook creates everything fresh. It stops if the stack `a2a-enterprise-identity` or
the secret `a2a-enterprise/orchestrator-credentials` (including one scheduled for
deletion) already exists, or if `onboarding-request.json` or `onboarding-response.json` is
next to this notebook. In that case, run notebook 3's Clean up (every cell above E0 in
notebook 3, which are its Setup, Helpers, and Onboarding handoff helpers, then the Clean up
section with `KEEP_ONBOARDING` left False), or delete the files, then run this notebook again
from the top.

</details>

In [ ]:
with step("I0.1", "Preflight: the enterprise account, then leftovers from an earlier run",
          hint="For the wrong account, switch the profile in .env.enterprise (or use the enterprise notebook "
               "instance). For leftovers, run every cell above E0 in notebook 3 (Setup, Helpers, and Onboarding "
               "handoff helpers) and its full Clean up (see the note above), or delete the files named above. "
               "Then run this notebook again from the top."):
    refuse_other_side_account(("a2a-saas-identity", "a2a-saas-runtime"), ".env.saas")
    leftovers = [f"stack {IDENTITY_STACK}"] if stack_exists(IDENTITY_STACK) else []
    if secret_exists(ORCHESTRATOR_SECRET_NAME):
        leftovers.append(f"secret {ORCHESTRATOR_SECRET_NAME} (possibly scheduled for deletion)")
    for name in (REQUEST_FILE, RESPONSE_FILE):
        if (HANDOFF_DIR / name).exists():
            leftovers.append(f"local file {name} (from an earlier run: run notebook 3's Clean up, or delete the file)")
    if leftovers:
        raise RuntimeError("Found resources from an earlier run: " + "; ".join(leftovers))
    log("No leftovers: ready to deploy")
    log("PASS I0.1: this is the enterprise account, and nothing is left from an earlier run")

## I1. Deploy the enterprise identity and reserve the callback API

Creates this account's Cognito user pool, its two app clients, the empty orchestrator secret, and the
partner (callback) API with no routes and no stage.

Pass when: the stack is created, and the step logs `PASS I1`.

<details><summary>Details</summary>

`cloudformation/enterprise-identity.yaml` creates:

- user pool `a2a-enterprise-pool`, with no users and no self sign-up, and the hosted domain
  `a2a-enterprise-<account ID>` that serves `/oauth2/token`
- resource server `a2a-enterprise-compliance-saas` (scope `invoke`): "call the compliance SaaS". The
  SaaS runtime will accept only this scope, from this pool, for one client.
- resource server `a2a-enterprise-orchestrator` (scope `ask`): the orchestrator's user API
- app clients `a2a-enterprise-orchestrator-saas` (the orchestrator, calling the SaaS) and
  `a2a-enterprise-orchestrator-user` (callers of the user API), both client credentials with 15-minute
  access tokens; their secrets never leave this account
- the empty secret `a2a-enterprise/orchestrator-credentials`
- the partner API `a2a-enterprise-context-api`, whose URL is the callback URL the SaaS provider registers

Every request to the partner API gets API Gateway's 404 until notebook 3 attaches the JWT
authorizer, the routes, and the stage. No client secret is a stack output.

</details>

In [ ]:
with step("I1", "Deploy the enterprise identity stack (a2a-enterprise-identity)",
          hint="The failed resources are printed above. A domain-prefix conflict means an earlier "
               "a2a-enterprise pool still exists in this Region: run notebook 3's Clean up."):
    identity_outputs = deploy_stack(IDENTITY_STACK, "cloudformation/enterprise-identity.yaml",
                                    {"NamePrefix": NAME_PREFIX}, max_minutes=10)
    ENT_POOL_ID = identity_outputs["UserPoolId"]
    ISSUER = identity_outputs["Issuer"]
    DISCOVERY_URL = identity_outputs["DiscoveryURL"]
    TOKEN_ENDPOINT = identity_outputs["TokenEndpoint"]
    ORCH_SAAS_CLIENT_ID = identity_outputs["OrchestratorSaasClientId"]
    ORCH_SAAS_SCOPE = identity_outputs["OrchestratorSaasScope"]
    ORCH_USER_CLIENT_ID = identity_outputs["OrchestratorUserClientId"]
    ORCH_USER_SCOPE = identity_outputs["OrchestratorUserScope"]
    ORCHESTRATOR_SECRET_ARN = identity_outputs["OrchestratorSecretArn"]
    CALLBACK_URL = identity_outputs["CallbackURL"]
    log(f"User pool:          {ENT_POOL_ID} (a2a-enterprise-pool)")
    log(f"Issuer:             {ISSUER}")
    log(f"Token endpoint:     {TOKEN_ENDPOINT}")
    log(f"Orchestrator-saas:  {ORCH_SAAS_CLIENT_ID} (scope {ORCH_SAAS_SCOPE})")
    log(f"Orchestrator-user:  {ORCH_USER_CLIENT_ID} (scope {ORCH_USER_SCOPE})")
    log(f"Callback URL:       {CALLBACK_URL} (reserved: no routes or stage yet; notebook 3 attaches the gate)")
    log(f"PASS I1: stack {IDENTITY_STACK} created; the callback URL is reserved (no routes or stage yet)")

## I2. Store the orchestrator's credentials

Reads the `a2a-enterprise-orchestrator-saas` client secret from Cognito in memory, writes it to
`a2a-enterprise/orchestrator-credentials`, and prints nothing.

Pass when: the step logs `PASS I2`.

<details><summary>Details</summary>

The orchestrator calls the SaaS agent with tokens from this account's pool, so its client secret
stays in this account. The cell reads it with `describe_user_pool_client`, writes
`{client_id, client_secret, token_endpoint, scope}` with `put_secret_value`, and deletes it from
memory.

Until notebook 3 attaches the secret's resource policy (E4), any principal in this account
whose IAM policy allows `secretsmanager:GetSecretValue` on it, for example an
administrator, can read it. After E4, the resource policy denies `GetSecretValue` to every
principal except the orchestrator's task role, which notebook 3 creates. No notebook reads
it back.

</details>

In [ ]:
with step("I2", "Store the orchestrator's credentials in a2a-enterprise/orchestrator-credentials",
          hint="The notebook role needs cognito-idp:DescribeUserPoolClient and secretsmanager:PutSecretValue."):
    _client = cognito.describe_user_pool_client(UserPoolId=ENT_POOL_ID, ClientId=ORCH_SAAS_CLIENT_ID)["UserPoolClient"]
    secrets.put_secret_value(SecretId=ORCHESTRATOR_SECRET_ARN, SecretString=json.dumps({
        "client_id": ORCH_SAAS_CLIENT_ID,
        "client_secret": _client["ClientSecret"],
        "token_endpoint": TOKEN_ENDPOINT,
        "scope": ORCH_SAAS_SCOPE,
    }))
    del _client
    log(f"Stored in {ORCHESTRATOR_SECRET_NAME}; the orchestrator task role (notebook 3) becomes its only reader "
        f"once E4 attaches the resource policy; the secret is not in memory")
    log(f"PASS I2: the orchestrator's credentials are stored in {ORCHESTRATOR_SECRET_NAME}; nothing was printed")

## I3. OAuth smoke test

Gets a client-credentials token for each enterprise client and checks the claims an authorizer checks.
Neither the secrets nor the tokens are printed.

Pass when: both tokens carry the expected `client_id`, `scope`, and `token_use`, and the step logs
`PASS I3`.

<details><summary>Details</summary>

The tokens are requested the way the orchestrator and the user API's callers will request them, and
only their claims are printed. A new Cognito prefix domain can take a minute or two before its token
endpoint answers, so the request retries.

</details>

In [ ]:
with step("I3", "OAuth smoke test for both enterprise clients",
          hint="invalid_client or invalid_scope means the client or scope is wrong; a connection error means "
               "the new domain is not live yet: wait a minute and re-run. The notebook role needs "
               "cognito-idp:DescribeUserPoolClient and DescribeUserPoolDomain."):
    wait_for_cognito_domain(f"{NAME_PREFIX}-{ACCOUNT_ID}")
    for _name, _client_id, _scope in (("a2a-enterprise-orchestrator-saas", ORCH_SAAS_CLIENT_ID, ORCH_SAAS_SCOPE),
                                      ("a2a-enterprise-orchestrator-user", ORCH_USER_CLIENT_ID, ORCH_USER_SCOPE)):
        _token, _expires_in = own_client_token(ENT_POOL_ID, _client_id, TOKEN_ENDPOINT, _scope, attempts=10)
        _claims = check_token_claims(_token, _client_id, _scope)
        del _token
        log(f"  {_name}: client_id={_claims['client_id']}, scope={_claims['scope']}, "
            f"token_use={_claims['token_use']}, expires in {_expires_in}s")
    log("PASS I3: both enterprise clients got tokens with the expected client_id, scope, and token_use")

## I4. Write the onboarding request

Builds `onboarding-request.json` from the stack outputs, checks it the way notebook 2 will, writes it
next to this notebook, and shows the same values as one highlighted line.

Pass when: the step logs `PASS I4`. Copy the highlighted line (or give the file) to the SaaS provider;
it holds public values only.

<details><summary>Details</summary>

The check includes a fetch of this pool's OIDC discovery document. The fields:

| Field | What it is | What the SaaS provider does with it |
|---|---|---|
| `issuer`, `discovery_url` | this pool's public OIDC issuer and discovery document | the runtime's JWT authorizer `discoveryUrl` |
| `token_endpoint` | this pool's token URL | advertised on the SaaS Agent Card; the SaaS never calls it |
| `client_id` | the `a2a-enterprise-orchestrator-saas` client ID (an identifier, not a credential) | the authorizer's `allowedClients` |
| `scope` | `a2a-enterprise-compliance-saas/invoke` | the authorizer's `allowedScopes` |
| `callback_url` | the reserved partner API URL | the only address the SaaS agent calls back |

In production this goes through the SaaS provider's onboarding portal or API.

</details>

In [ ]:
with step("I4", "Write the onboarding request (public values only)",
          hint="A failed self-check right after I1 usually means the new pool is not answering yet: wait a minute "
               "and re-run this cell."):
    ONBOARDING_REQUEST = {
        "kind": REQUEST_KIND,
        "version": 1,
        "issuer": ISSUER,
        "discovery_url": DISCOVERY_URL,
        "token_endpoint": TOKEN_ENDPOINT,
        "client_id": ORCH_SAAS_CLIENT_ID,
        "scope": ORCH_SAAS_SCOPE,
        "callback_url": CALLBACK_URL,
    }
    validate_request(ONBOARDING_REQUEST, None)
    check_discovery(ISSUER, TOKEN_ENDPOINT)
    write_handoff(REQUEST_FILE, ONBOARDING_REQUEST)
    log("PASS I4: onboarding-request.json is written and passes the checks notebook 2 runs (public values only)")

import html
from IPython.display import HTML, display
display(HTML(
    "<p><b>Next: copy the highlighted text below. You need it in notebook 2 (2_saas_provider.ipynb, in the SaaS "
    "provider account), saved as onboarding-request.json.</b></p>"
    "<pre style='background:#fff3b0;border:2px solid #d4a017;padding:10px;white-space:pre-wrap;word-break:break-all'>"
    f"{html.escape(json.dumps(ONBOARDING_REQUEST, separators=(',', ':')))}</pre>"
    "<p>It contains no secret. In production this goes through the SaaS provider's onboarding portal or API.</p>"
))

## Reference

Open a section when you need it. The README has the full versions.

<details><summary>Where to run</summary>

A SageMaker AI notebook instance in the enterprise account (use the same one for notebook 3), or a local Jupyter
with this account's profile in `.env.enterprise` in the project folder, next to the notebooks
(`AWS_PROFILE=...`, and optionally `AWS_REGION=...`; the notebook reads it before `.env`). Either
way, the kernel needs Python 3.10 or later. On a SageMaker AI notebook instance, choose the
`conda_python3` kernel. For local runs, use IAM Identity Center profiles (`aws configure sso`), so
the deployer works with temporary credentials. The notebook finds its project folder by its files,
so the unzipped folder can have any name.

</details>

<details><summary>Deployer permissions</summary>

- Deployer permissions for the notebook's role. A sandbox administrator role covers them:
  - `cloudformation:*` on the stack `a2a-enterprise-identity`, plus the Cognito actions,
    API Gateway `POST`, `GET`, and `DELETE` on `/apis`, and Secrets Manager `CreateSecret`
    that CloudFormation needs to create it
  - `cognito-idp:DescribeUserPoolClient` and `cognito-idp:DescribeUserPoolDomain`
  - Secrets Manager `DescribeSecret` and `PutSecretValue`
  - `bedrock:InvokeModel` for the preflight, plus `aws-marketplace:Subscribe`,
    `aws-marketplace:Unsubscribe`, and `aws-marketplace:ViewSubscriptions`, which the first
    model call in the account needs (the workload roles don't)
  - no `iam:PassRole`: this notebook creates no role

</details>

<details><summary>What this notebook creates</summary>

| Resource | Name | Created by |
|---|---|---|
| Identity stack | `a2a-enterprise-identity`: user pool `a2a-enterprise-pool` (no users), domain `a2a-enterprise-<account ID>`, resource servers `a2a-enterprise-compliance-saas` (scope `a2a-enterprise-compliance-saas/invoke`, "call the compliance SaaS") and `a2a-enterprise-orchestrator` (scope `a2a-enterprise-orchestrator/ask`, the user API), app clients `a2a-enterprise-orchestrator-saas` and `a2a-enterprise-orchestrator-user` (client credentials, 15-minute access tokens), empty secret `a2a-enterprise/orchestrator-credentials` | I1 (CloudFormation) |
| Reserved partner (callback) API | `a2a-enterprise-context-api`, an API Gateway HTTP API with no routes and no stage yet, so its URL is known before the SaaS side is set up and nothing is reachable through it | I1 (same stack) |
| Secret value | the `a2a-enterprise-orchestrator-saas` client's credentials in `a2a-enterprise/orchestrator-credentials`, for the orchestrator task role that notebook 3 creates | I2 |
| Local file | `onboarding-request.json`, next to this notebook | I4 |

This notebook has no Clean up section. Notebook 3's Clean up removes everything notebooks 1
and 3 created, and works even if notebook 3 never deployed anything.

</details>

<details><summary>Security in this demo</summary>

| Path | Protocol and check |
|---|---|
| Orchestrator to compliance agent (AgentCore) | HTTPS. AgentCore validates a JWT from the enterprise pool: `discoveryUrl` (the issuer), `allowedClients` (the token's `client_id` must be `a2a-enterprise-orchestrator-saas`), and `allowedScopes` |
| Compliance agent to context agent (partner API) | HTTPS, only to the callback URL registered at onboarding, after the agent checks the public context card there. API Gateway validates a JWT from the SaaS pool: issuer, the `a2a-saas-callback-client` client ID, and the route scope |
| Caller to orchestrator (user API) | HTTPS. API Gateway validates a JWT from the enterprise pool: issuer, the `a2a-enterprise-orchestrator-user` client ID, and the route scope. API Gateway also overwrites a caller header with the token's `client_id`, and a job is visible only to the client that submitted it |
| Inside the enterprise VPC | HTTP from the VPC link to the internal NLB to the tasks, in private subnets with no public IPs. Security groups chain VPC link, then NLB, then tasks; nothing else can reach the agent ports. TLS terminates at API Gateway. Production: a TLS listener on the NLB with an ACM certificate, or TLS in the containers, if your policy requires encryption inside the VPC |

- No client secret, key, or IAM permission crosses between the accounts. Only two files of
  public values and HTTPS calls with 15-minute access tokens do.
- Each secret is readable by exactly one workload role. The deployer reads app client
  secrets from Cognito, in memory only, to write the secret and to get test tokens. A
  resource policy on each secret denies `GetSecretValue` to every other principal, and no
  notebook reads a value back from Secrets Manager. The enterprise secret gets its
  resource policy in notebook 3 (E4); until then, only IAM identity policies limit who in
  the enterprise account can read it.
- No notebook output contains a client secret or an access token, and no secret is ever
  typed. The CloudFormation secrets are created empty, so a stack update can never
  overwrite a value the notebook wrote.
- Every HTTP call in the notebooks refuses redirects, so a 3xx can't move a credential to
  another host.
- Every IAM policy names exact ARNs. Bedrock access names one inference-profile ARN in
  this account and Region, plus the model's foundation-model ARN with only the Region
  segment wildcarded (the `us.` profile can route a request to another US Region).
  `*` appears only for actions with no resource-level support, each with a comment in
  the template.
- Before anything is reported as working, each notebook checks that anonymous calls to
  the gated endpoints are denied.
- Every model call carries the account's Amazon Bedrock guardrail (a prompt-attack filter and
  harmful-content filters, no sensitive-information filter). Text from the other organization
  that a model call can't evaluate, a tool result or an Agent Card, is screened with
  `ApplyGuardrail` before any model sees it. S3.1 and E4.1 check each guardrail with fixed texts.

</details>

<details><summary>Cost estimate</summary>

This notebook's resources (a user pool with no users, one secret, and an API with no stage) have
almost no idle cost. The enterprise account's full table is in notebook 3's Reference and in the
README (Cost estimate).

</details>

<details><summary>If a step failed: retry this notebook</summary>

Every step refuses resources that already exist, so a failed step can't be fixed by running its cell
again. Notebook 3's Clean up removes everything notebooks 1 and 3 created, and works even if notebook 3
never deployed: run every cell above E0 in notebook 3 (Setup, Helpers, and Onboarding handoff helpers),
then its Clean up with `KEEP_ONBOARDING` left False. Then run this notebook again from the top.

</details>

<details><summary>After a kernel restart</summary>

Run every cell above I0 (Setup, Helpers, and Onboarding handoff helpers) first. The steps keep their
values only in the kernel, so after a restart in the middle of this notebook, follow If a step failed.

</details>

<details><summary>Recent logs</summary>

This notebook creates no log groups. A failed stack prints its failed resources in the step's output.

</details>

## Next

1. The SaaS provider runs `2_saas_provider.ipynb` in its account with `onboarding-request.json` (or
   the highlighted line I4 showed) and sends back `onboarding-response.json`.
2. Put `onboarding-response.json` next to the notebooks here, or keep its line for the paste, then
   run `3_enterprise_agents.ipynb` in this account.

This notebook has no Clean up section: notebook 3's Clean up removes everything notebooks 1 and 3
created.

Next: open 2_saas_provider.ipynb in the SaaS provider account.